# SETUP

In [1]:
import os
import sys
sys.path.insert(0,'/root/eallmancy')
sys.path.insert(0,'/root/eallmancy/experiments')
os.chdir('/root/eallmancy')

from eallmancy.engine.llm_solution_engine_v3 import EngineV3
from eallmancy import Artifact
from eallmancy import SinglePartSolution, ProblemSpec
from eallmancy.engine.engine_trace import EngineTrace
from collections import defaultdict
import datetime

import os
import re

# TODO: THIS IS ULTRA HACKY, PUT API SETTINGS IN CONTROL PANEL
with open('/root/openai_api_key') as f:
    os.environ['OPENAI_API_KEY'] = f.read().replace('\n','')
with open('/root/gemini_api_key') as f:
    os.environ['GEMINI_API_KEY'] = f.read().replace('\n','')


# from experiments.scievo import SciEvo
from experiments.toyeoh import ToyEoH
from problems.llm4ad import (    
    CarMountainProblemSpec,
    PendulumProblemSpec,

    
    StressTrainProblemSpec,
    BactGrowProblemSpec,
    
    AdmissibleSetProblemSpec,
    VRPTWProblemSpec
)

2025-07-23 21:41:18.835460: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753306878.860869 1513653 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753306878.867840 1513653 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1753306878.886477 1513653 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1753306878.886523 1513653 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1753306878.886525 1513653 computation_placer.cc:177] computation placer alr

# sim func

In [2]:
CHATGPT_4OMINI_API = dict(
            host="api.openai.com",
            key=os.environ.get('OPENAI_API_KEY'),
            model="gpt-4o-mini",
            timeout=20
        )

GEMINI_25FLASH_API = dict(
            base_url='https://generativelanguage.googleapis.com/v1beta',
            api_key=os.environ.get('GEMINI_API_KEY'),
            model='gemini-2.0-flash',
            timeout=20,
            openai_format=True
        )

## FS


In [3]:
%%prun -l reevo.py



# from algorithms.funsearch import funsearch
# algo = funsearch

# from algorithms.eoh import eoh
# algo = eoh

from algorithms.reevo import reevo
algo = reevo

# def funsearch(n_isles=10, max_time=2*60, reset_period=4*60*60, api_spec=None, problem_spec=None):

ps = CarMountainProblemSpec(maximization=True)
# ps = StressTrainProblemSpec(maximization=True)
temp = 0.75
llm_api = CHATGPT_4OMINI_API

trace_res = algo(problem_spec=ps, api_spec=dict(temperature=temp, **llm_api), max_samples=10, pop_size=6) # max_time in (s)... probably need to change
    

/root/reallm/venv/lib/python3.12/site-packages/gym/utils/passive_env_checker.py:233: DeprecationWarning: `np.bool8` is a deprecated alias for `np.bool_`.  (Deprecated NumPy 1.24)
  if not isinstance(terminated, (bool, np.bool8)):


         214637 function calls (202815 primitive calls) in 101.500 seconds

   Ordered by: internal time
   List reduced from 882 to 15 due to restriction <'reevo.py'>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        3    0.000    0.000    3.286    1.095 reevo.py:64(comparative_reflection)
        3    0.000    0.000   21.773    7.258 reevo.py:81(crossover_solution)
        1    0.000    0.000   98.525   98.525 reevo.py:129(reevo)
        6    0.000    0.000   44.809    7.468 reevo.py:54(initial_solution)
        3    0.000    0.000    0.002    0.001 reevo.py:34(__init__)
        1    0.000    0.000    0.000    0.000 reevo.py:1(<module>)
        3    0.000    0.000   27.375    9.125 reevo.py:109(elite_mutated_solution)
        6    0.000    0.000    0.000    0.000 reevo.py:173(<lambda>)
        1    0.000    0.000    1.282    1.282 reevo.py:98(longterm_reflection)
        1    0.000    0.000    0.000    0.000 reevo.py:42(__init__)
        3    0.000    0

In [4]:
print(ps.template)



import numpy as np

def choose_action(pos: float, v: float, last_action: int) -> int:
    """Return the action for the car to proceed the next move.
    Args:
        pos: Car's position, a float ranges between [-1.2, 0.6].
        v: Car's velocity, a float ranges between [-0.07, 0.07].
        last_action: Car's next move, a int ranges between [0, 1, 2].
    Return:
         An integer representing the selected action for the car.
         0: accelerate to left
         1: don't accelerate
         2: accelerate to right
    """
    return np.random.randint(3)



In [5]:
print(trace_res[0])



Artifact 0
```
import numpy as np

def choose_action(pos: float, v: float, last_action: int) -> int:
    """Return the action for the car to proceed the next move.
    Args:
        pos: Car's position, a float ranges between [-1.2, 0.6].
        v: Car's velocity, a float ranges between [-0.07, 0.07].
        last_action: Car's last move, an int ranges between [0, 1, 2].
    Return:
         An integer representing the selected action for the car.
         0: accelerate to left
         1: don't accelerate
         2: accelerate to right
    """
    
    # Define the target position
    target_pos = 0.6
    
    # Calculate the distance to the target
    distance_to_target = target_pos - pos
    
    # Determine the action based on position and velocity
    if distance_to_target > 0:
        if v < 0:  # If the car is moving backward
            action = 2  # Accelerate to the right to move forward
        elif v > 0 and distance_to_target > 0.1:  # Moving forward but not too close t

In [6]:
[
    a.fitness
    for a in trace_res.list if a.is_solution
]


[-1.8007745444774628,
 -1.8632910251617432,
 -2.021572768688202,
 -1.9584235846996307,
 -2.025023400783539,
 -1.873861312866211,
 -1.7814607620239258,
 -1.7244784832000732,
 -1.958618551492691,
 -1.9568616151809692,
 -1.9576306343078613,
 -1.7721136212348938]